PASO 5.1| FEATURE ENGINEERING.

--------------------------------

NUESTRO MEJOR ALIADO PARA CAPTURAR UNA MEJOR CORRELACION LINEAL Y NO LINEAL ENTRE VARIABLES Y VARIABLE TARGET
ES ENTENDER LAS POSIBILIDADES QUE OFRECEN LOS DATOS DE LAS DIFRENTES DISTRIBUCIONES DE LAS VARIABLES
Y LAS CORRELACIONES LINEALES ENTRE VARIABLES GRACIAS AL MAPA DE CALOR.

SE TRATA DE VISUALIZAR CUANDO UNA DISTRIBUCION ES POSIBLE DERIVARLA EN DOS DADO SU SEGREGACION EN DOS O MAS GRANDES GRUPOS DE MUESTRAS.

TAMBIEN SE TRATA DE ENTENDER QUE CUANDO UNA VARIABLE TIENE MUCHA CORRELACION ENTRE OTRAS SE PUEDEN COMBINAR, INCLUSIO HACER MEDIAS ARITMETICAS Y SENSIBILIZAR LA VARIABLE PRODUCTO CAMBIANDOLA DE DISCRETA A CONTINUA.
BINARIZAR TAMBIEN ES UNA OPCION.

INCLUSO PASAR DE UNA VARIABLE CONTINUA A DISCONTINUA SI SU CORRELACION ES BAJA PARA CAPTURAR OTRAS CORRELACIONES NO LINEALES.

In [2]:
import numpy as np, random
import pandas as pd

In [3]:
df1 = pd.read_csv('C:/Users/Josue/4GA.Datascience/4GA.DataScience/data/processed/muestra_12k.csv')

In [4]:
#Empezamos por binarizacion:
#Observamos distribuciones y variables con poca linealidad.
#Si es simetrica gaussiana se hace un cut, si es simetrica muy desplazada se binariza, ponderizando
#en funcion del diferencial de muestras de los dos primeros cuartiles respecto a los dos ultimos.
import pandas as pd

def factorizar_religiosidad(df, columna='rlgdgr'):

  
    def asignar_categoria(x):
        if x <= 3.9:
            return 0  # ateo
        elif 4 <= x <= 6:
            return 1  # escéptico
        else:
            return 2  # creyente

    df['rel3fc'] = df[columna].apply(asignar_categoria)
    return df

# Ejemplo de uso:
df1 = factorizar_religiosidad(df1)
print(df1['rel3fc'].value_counts())


rel3fc
0    4870
1    4042
2    3888
Name: count, dtype: int64


In [5]:
print(df1['happy'].value_counts())

happy
8     3827
7     2623
9     2007
6     1212
5     1110
10    1089
4      387
3      293
2      129
0       62
1       61
Name: count, dtype: int64


In [ ]:
#18-'happy': Escala de 0 a 10, grado de cuan feliz eres, siendo 0 infeliz.
#Apenas tiene linealidad +/- vamos a derivarla en dos nuevas variables. A ver si capturamos correlaciones
# unhappybin y happybin
def pondbin(df, columna='happy'):

    q1 = df[columna].quantile(0.25)
    q3 = df[columna].quantile(0.75)

    def asignar_categoria(x):
        if x < q1:
            return 'bajo'
        elif x > q3:
            return 'alto'
        else:  # q1 <= x <= q3
            if x == 5:
                if abs(5 - q1) < abs(5 - q3):
                    return 'bajo'  # Ponderar hacia 'bajo'
                else:
                    return 'alto'  # Ponderar hacia 'alto'
            else:
                return 'medio'

    df['happyfc'] = df[columna].apply(asignar_categoria)
    return df
df1 = pondbin(df1)
print(df1['happyfc'].value_counts())


happyfc
medio    7662
alto     3096
bajo     2042
Name: count, dtype: int64


In [ ]:
dicthappy = {
    'bajo': '0',
    'medio': '1',
    'alto': '2',
}
df1['happyfc'] = df1['happyfc'].map(dicthappy)
print(df1['happyfc'].value_counts())

happyfc
1    7662
2    3096
0    2042
Name: count, dtype: int64


In [16]:

def escalacontinuamedia(df):
    columnas_confianza = [
        'trstep', 'trstlgl', 'trstplc', 'trstplt', 'trstprl', 'trstprt', 'trtsci_pnd'
    ]

    df['confianza_promedio'] = df[columnas_confianza].mean(axis=1)
    return df

df1 = escalacontinuamedia(df1)
print(df1['confianza_promedio'].describe())

count    12800.000000
mean         4.946283
std          1.697562
min          0.000000
25%          3.714286
50%          5.000000
75%          6.285714
max         10.000000
Name: confianza_promedio, dtype: float64


In [ ]:

import pandas as pd
def cutcontinua(df, columna, bins):
    if columna not in df.columns:
        raise ValueError(f"La columna '{columna}' no existe en el DataFrame.")

    if bins <= 0:
        raise ValueError("El número de bins debe ser mayor que 0.")

    labels = list(range(bins))  # Etiquetas numéricas (0, 1, 2, ...)
    df[columna + '_factorizada'] = pd.cut(df[columna], bins=bins, labels=labels, right=False)
    return df

df1 = cutcontinua(df1, 'confianza_promedio', bins=5)
print(df1['confianza_promedio_factorizada'].value_counts())

confianza_promedio_factorizada
2    5266
3    3650
1    2974
0     594
4     316
Name: count, dtype: int64


In [20]:
def escalacontinuamedia(df):
    columnas_satisfecho = [
        'stfgov', 'stfeco', 'stfdem',
    ]

    df['satisf_media'] = df[columnas_satisfecho].mean(axis=1)
    return df

df1 = escalacontinuamedia(df1)
print(df1['satisf_media'].describe())

count    12800.000000
mean         4.626719
std          2.055460
min          0.000000
25%          3.333333
50%          4.666667
75%          6.333333
max         10.000000
Name: satisf_media, dtype: float64


In [21]:
def cutcontinua(df, columna, bins):
    if columna not in df.columns:
        raise ValueError(f"La columna '{columna}' no existe en el DataFrame.")

    if bins <= 0:
        raise ValueError("El número de bins debe ser mayor que 0.")

    labels = list(range(bins))  # Etiquetas numéricas (0, 1, 2, ...)
    df[columna + '_factorizada'] = pd.cut(df[columna], bins=bins, labels=labels, right=False)
    return df

df1 = cutcontinua(df1, 'satisf_media', bins=5)
print(df1['satisf_media_factorizada'].value_counts())

satisf_media_factorizada
2    4464
3    3470
1    2975
0    1370
4     521
Name: count, dtype: int64


In [22]:
def pondbin(df, columna):

    q1 = df[columna].quantile(0.25)
    q3 = df[columna].quantile(0.75)

    def asignar_categoria(x):
        if x < q1:
            return 'bajo'
        elif x > q3:
            return 'alto'
        else:  # q1 <= x <= q3
            if x == 5:
                if abs(5 - q1) < abs(5 - q3):
                    return 'bajo'  # Ponderar hacia 'bajo'
                else:
                    return 'alto'  # Ponderar hacia 'alto'
            else:
                return 'medio'

    df['pintfc'] = df[columna].apply(asignar_categoria)
    return df
df1 = pondbin(df1, 'polintr')
print(df1['pintfc'].value_counts())

pintfc
medio    9248
alto     1850
bajo     1702
Name: count, dtype: int64


In [23]:
dictint = {
    'bajo': '0',
    'medio': '1',
    'alto': '2',
}
df1['pintfc'] = df1['pintfc'].map(dicthappy)
print(df1['pintfc'].value_counts())

pintfc
1    9248
2    1850
0    1702
Name: count, dtype: int64


In [24]:
def escalacontinuamedia(df):
    columnas_gente= [
        'pplfair', 'pplhlp', 'ppltrst',
    ]

    df['ppl'] = df[columnas_gente].mean(axis=1)
    return df

df1 = escalacontinuamedia(df1)
print(df1['ppl'].describe())

count    12800.000000
mean         5.099818
std          1.826857
min          0.000000
25%          4.000000
50%          5.333333
75%          6.333333
max         10.000000
Name: ppl, dtype: float64


In [25]:
def cutcontinua(df, columna, bins):
    if columna not in df.columns:
        raise ValueError(f"La columna '{columna}' no existe en el DataFrame.")

    if bins <= 0:
        raise ValueError("El número de bins debe ser mayor que 0.")

    labels = list(range(bins))  # Etiquetas numéricas (0, 1, 2, ...)
    df[columna + '_fc'] = pd.cut(df[columna], bins=bins, labels=labels, right=False)
    return df

df1 = cutcontinua(df1, 'ppl', bins=4)
print(df1['ppl_fc'].value_counts())

ppl_fc
2    6583
1    4109
0    1114
3     994
Name: count, dtype: int64


In [26]:
def cutcontinua(df, columna, bins):
    if columna not in df.columns:
        raise ValueError(f"La columna '{columna}' no existe en el DataFrame.")

    if bins <= 0:
        raise ValueError("El número de bins debe ser mayor que 0.")

    labels = list(range(bins))  # Etiquetas numéricas (0, 1, 2, ...)
    df[columna + '_fc'] = pd.cut(df[columna], bins=bins, labels=labels, right=False)
    return df

df1 = cutcontinua(df1, 'lrscale', bins=4)
print(df1['lrscale_fc'].value_counts())

lrscale_fc
2    6629
1    2932
0    1752
3    1487
Name: count, dtype: int64


In [27]:
def cutcontinua(df, columna, bins):
    if columna not in df.columns:
        raise ValueError(f"La columna '{columna}' no existe en el DataFrame.")

    if bins <= 0:
        raise ValueError("El número de bins debe ser mayor que 0.")

    labels = list(range(bins))  # Etiquetas numéricas (0, 1, 2, ...)
    df[columna + '2_fc'] = pd.cut(df[columna], bins=bins, labels=labels, right=False)
    return df

df1 = cutcontinua(df1, 'lrscale', bins=5)
print(df1['lrscale2_fc'].value_counts())

lrscale2_fc
2    5497
3    2573
1    2395
4    1487
0     848
Name: count, dtype: int64


In [28]:
df1.to_csv('C:/Users/Josue/4GA.Datascience/4GA.DataScience/models/presplit.csv')

Seguimos en otro documento para tantear en diferentes modelos.